# House Price Prediction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.impute  import SimpleImputer
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from skopt import BayesSearchCV

pd.set_option('display.max_rows', None)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
train_data = pd.read_csv(r"train.csv")
train_data.head(5)

In [ ]:
train_data.isna().sum()

In [34]:
test_data = pd.read_csv("test.csv")
test_data.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.00,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.00,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [ ]:
test_data.shape

In [ ]:
test_data.isna().sum()

### Remove Duplicate values(check Id)

In [ ]:
duplicates = train_data['Id'].duplicated()
train_data[duplicates]

In [ ]:
train_data.drop_duplicates(inplace=True)

### Remove values with no Id

In [ ]:
no_id_data = train_data[train_data['Id'].isna()]
no_id_data

In [ ]:
train_data = train_data[train_data['Id'].notna()]
train_data.head(5)

### Target Variables 

In [ ]:
x = train_data.drop(columns = ['SalePrice' , 'Id'])
y = train_data['SalePrice']

#### Transform Y using yeo-jhonson transformation

In [ ]:
y_transformer = PowerTransformer(method='yeo-johnson' , standardize=True)
y_transformed = y_transformer.fit_transform(y.to_frame())

### Important Variables

In [ ]:
ordinal_cols = ['LotShape','Utilities','LandSlope','ExterQual','ExterCond',
                'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','HeatingQC','KitchenQual','Functional',
                'FireplaceQu','GarageFinish','GarageQual','GarageCond','PavedDrive','PoolQC','Fence']



nominal_cols = ['MSZoning','Street','Alley','Neighborhood','LotConfig','BldgType','Condition1','Condition2','HouseStyle','LandContour','RoofStyle','RoofMatl','Exterior1st' ,'Exterior2nd' ,
                'MasVnrType' , 'Foundation' , 'Heating' , 'GarageType' , 'MiscFeature' , 'SaleType' , 'SaleCondition', 'Electrical' , 'CentralAir']



impute_constant_cols = ['MSSubClass','MSZoning','Street','LotShape','LandContour','Utilities','LotConfig','LandSlope',
                        'Neighborhood','Condition1','Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st',
                        'Exterior2nd','ExterQual','ExterCond','Foundation','Heating','HeatingQC','CentralAir','Electrical',
                        'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual','Fireplaces','GarageCars',
                        'PavedDrive','SaleType','SaleCondition', 'Functional']


# cols whose na values can be filled with median
numerical_cols = ['MSSubClass','LotArea','LotFrontage','OverallQual','OverallCond','YearBuilt','YearRemodAdd',
                  'MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','1stFlrSF','2ndFlrSF','LowQualFinSF',
                  'GrLivArea','BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr','KitchenAbvGr','TotRmsAbvGrd'
                  ,'Fireplaces','GarageYrBlt','GarageCars','GarageArea','WoodDeckSF','OpenPorchSF','EnclosedPorch','3SsnPorch'
                  ,'ScreenPorch','PoolArea','MiscVal','MoSold','YrSold']


# cols whose na values can be filled with default text values (like 'None' or 'No Garage')
# cols whose na values can be filled with 0
# (GarageYrBlt is kept here because if a house has no garage, a 0 is a safe placeholder before scaling/binarizing)
fill_default_cols = ['Alley','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageType',
                        'GarageFinish','GarageQual','GarageCond','PoolQC','Fence','MiscFeature' ,'GarageYrBlt']


default_map = {
    'Alley' : 'No alley',
    'MasVnrType' : 'None',
    'BsmtQual' : 'No Basement',
    'BsmtCond' : 'No Basement',
    'BsmtExposure' : 'No Basement',
    'BsmtFinType1' : 'No Basement',
    'BsmtFinType2' : 'No Basement',
    'FireplaceQu' : 'No Fireplace',
    'GarageType' : 'No Garage',
    'GarageFinish' : 'No Garage',
    'GarageQual' : 'No Garage',
    'GarageCond' : 'No Garage',
    'GarageYrBlt' : 0,
    'PoolQC' : 'No Pool',
    'Fence' : 'No Fence',
    'MiscFeature' : 'None'
}


ordinal_categories = {
'LotShape' : ['Missing' , 'IR3' , 'IR2' , 'IR1' , 'Reg'],
'Utilities' : ['Missing' , 'ELO' , 'NoSeWa' , 'NoSewr', 'AllPub'],
'LandSlope' : ['Missing' , 'Sev' , 'Mod' , 'Gtl'],
'ExterQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'ExterCond' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtQual' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtCond' : ['No Basement' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'BsmtExposure' : ['No Basement' , 'No' , 'Mn' , 'Av' , 'Gd'],
'BsmtFinType1' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'BsmtFinType2' : ['No Basement' , 'Unf' , 'LwQ' , 'Rec' , 'BLQ' , 'ALQ' , 'GLQ'],
'HeatingQC' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'KitchenQual' : ['Missing' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Functional' :['Missing', 'Sal' , 'Sev' , 'Maj2' , 'Maj1' , 'Mod' , 'Min2' , 'Min1' , 'Typ'],
'FireplaceQu' : ['No Fireplace' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageFinish' : ['No Garage' , 'Unf' , 'RFn' , 'Fin'],
'GarageQual' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'GarageCond' : ['No Garage' , 'Po' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'PavedDrive' : ['Missing' , 'N' , 'P' , 'Y'],
'PoolQC' : ['No Pool' , 'Fa' , 'TA' , 'Gd' , 'Ex'],
'Fence' : ['No Fence' , 'MnWw' , 'GdWo' , 'MnPrv' , 'GdPrv']
}


### Fill Default value columns 

In [ ]:
for col in fill_default_cols:
    default_val = default_map.get(col)
    train_data[col] = train_data[col].fillna(default_val)
    test_data[col] = test_data[col].fillna(default_val)
    print(f"Default Values of {col} filled with {default_val}")
    print(train_data[col].unique())
    print(train_data[col].unique())

In [ ]:
num_pipeline = Pipeline(
    steps=[
        ('imputer' , SimpleImputer(strategy='median')),
        ('scaler' , StandardScaler()) 
    ]
)

### Ordinal Pipeline

In [ ]:
categories = [ordinal_categories.get(col) for col in ordinal_categories.keys()]
categories

In [ ]:
ord_pipeline = Pipeline(
    steps=[
        ('imputer' , SimpleImputer(strategy='constant' , fill_value= 'Missing')),
        ('encoder' , OrdinalEncoder(categories= categories , handle_unknown= 'use_encoded_value' , unknown_value=-1))
    ]
)

### Nominal Pipeline

In [ ]:
nom_pipeline = Pipeline(
    steps= [
        ('imputer' , SimpleImputer(strategy='constant' , fill_value='Missing')),
        ('encoder' , OrdinalEncoder(handle_unknown='use_encoded_value' , unknown_value=-1))
    ]
)

### Preprocessing Pipeline

In [ ]:
preprocessing = ColumnTransformer(
    transformers= [
        ('num' , num_pipeline , numerical_cols),
        ('ord' , ord_pipeline , ordinal_cols),
        ('nom' , nom_pipeline , nominal_cols)
    ]
)

### Main Pipeline

In [ ]:
model_pipeline = Pipeline(
    steps=[
        ('preprocessing' , preprocessing),
        ('model' , LGBMRegressor(random_state=42 , device_type = 'cpu' , verbose = -1))
    ]
)

In [ ]:
num_cols_count = len(numerical_cols)
total_num_cols = x.shape[1]
cat_indics = list(range(num_cols_count,total_num_cols))

### Hyper Parameter Tuning

In [ ]:
kf = KFold(n_splits=5 , random_state=42 , shuffle=True)

In [ ]:
param_distributions = {
    # --- CORE SPEED & COMPLEXITY ---
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'model__num_leaves': [31, 63, 127, 255],
    'model__max_depth': [-1, 5, 7, 9],
    'model__max_bin': [64, 128, 255, 512],
    'model__n_estimators' : [100 , 200 , 500 , 1000 , 2000 , 3000 , 5000],
    
    #model__ --- OVERFITTING SHIELDS & PRUNING ---
    'model__min_child_samples': [10, 20, 30, 50],
    'model__min_child_weight': [1e-3, 0.1, 1.0, 5.0, 10.0],
    'model__min_split_gain': [0.0, 0.1, 0.5, 1.0],
    'model__path_smooth': [0.0, 1.0, 5.0, 10.0],
    
    #model__ --- REGULARIZATION ---
    'model__reg_alpha': [0, 0.1, 1.0, 5.0, 10.0],
    'model__reg_lambda': [0, 0.1, 1.0, 5.0, 10.0],
    
    #model__ --- STOCHASTIC SAMPLING ---
    'model__subsample': [0.6, 0.8, 1.0],
    'model__subsample_freq': [1, 3, 5, 7], 
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__colsample_bynode': [0.6, 0.8, 1.0]
}

In [ ]:
bayes_Search = BayesSearchCV(
    estimator= model_pipeline,
    search_spaces= param_distributions,
    n_iter=150,
    cv = kf,
    verbose=2,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    random_state=42
)

### Training and Finding Best Model

In [29]:
bayes_Search.fit(x , y_transformed.ravel() , model__categorical_feature = cat_indics)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV] END model__colsample_bynode=0.8, model__colsample_bytree=1.0, model__learning_rate=0.1, model__max_bin=512, model__max_depth=7, model__min_child_samples=50, model__min_child_weight=5.0, model__min_split_gain=0.0, model__n_estimators=200, model__num_leaves=127, model__path_smooth=1.0, model__reg_alpha=5.0, model__reg_lambda=0.1, model__subsample=0.8, model__subsample_freq=5; total time=   0.1s


c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[CV] END model__colsample_bynode=0.8, model__colsample_bytree=1.0, model__learning_rate=0.1, model__max_bin=512, model__max_depth=7, model__min_child_samples=50, model__min_child_weight=5.0, model__min_split_gain=0.0, model__n_estimators=200, model__num_leaves=127, model__path_smooth=1.0, model__reg_alpha=5.0, model__reg_lambda=0.1, model__subsample=0.8, model__subsample_freq=5; total time=   0.1s


c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[CV] END model__colsample_bynode=0.8, model__colsample_bytree=1.0, model__learning_rate=0.1, model__max_bin=512, model__max_depth=7, model__min_child_samples=50, model__min_child_weight=5.0, model__min_split_gain=0.0, model__n_estimators=200, model__num_leaves=127, model__path_smooth=1.0, model__reg_alpha=5.0, model__reg_lambda=0.1, model__subsample=0.8, model__subsample_freq=5; total time=   0.1s


...

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[CV] END model__colsample_bynode=1.0, model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_bin=128, model__max_depth=-1, model__min_child_samples=20, model__min_child_weight=1.0, model__min_split_gain=0.0, model__n_estimators=1000, model__num_leaves=63, model__path_smooth=1.0, model__reg_alpha=5.0, model__reg_lambda=1.0, model__subsample=1.0, model__subsample_freq=3; total time=   1.2s


c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[CV] END model__colsample_bynode=1.0, model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_bin=128, model__max_depth=-1, model__min_child_samples=20, model__min_child_weight=1.0, model__min_split_gain=0.0, model__n_estimators=1000, model__num_leaves=63, model__path_smooth=1.0, model__reg_alpha=5.0, model__reg_lambda=1.0, model__subsample=1.0, model__subsample_freq=3; total time=   1.5s


,estimator,Pipeline(step...verbose=-1))])
,search_spaces,"{'model__colsample_bynode': [0.6, 0.8, ...], 'model__colsample_bytree': [0.6, 0.8, ...], 'model__learning_rate': [0.01, 0.03, ...], 'model__max_bin': [64, 128, ...], ...}"
,optimizer_kwargs,None
,n_iter,150
,scoring,'neg_mean_squared_error'
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,KFold(n_split... shuffle=True)


In [30]:
print(f"Best Parametres : {bayes_Search.best_params_}")

Best Parametres : OrderedDict([('model__colsample_bynode', 0.8), ('model__colsample_bytree', 0.6), ('model__learning_rate', 0.05), ('model__max_bin', 512), ('model__max_depth', -1), ('model__min_child_samples', 20), ('model__min_child_weight', 10.0), ('model__min_split_gain', 0.1), ('model__n_estimators', 1000), ('model__num_leaves', 31), ('model__path_smooth', 5.0), ('model__reg_alpha', 0.1), ('model__reg_lambda', 1.0), ('model__subsample', 0.8), ('model__subsample_freq', 1)])


In [31]:
model = bayes_Search.best_estimator_

In [35]:
test_ids = test_data['Id']
test_data.drop(columns=['Id'] , inplace = True)

In [36]:
transformed_predictions = model.predict(test_data)

actual_predictions = y_transformer.inverse_transform(transformed_predictions.reshape(-1,1))
rounded_actual_predictions = [ round(x,4) for x in actual_predictions.flatten()]

c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\sohil\anaconda3\envs\AICourse\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(


In [37]:
results = pd.DataFrame({
    "Id" : test_ids,
    "SalePrice" : rounded_actual_predictions
})

results.to_csv("result.csv" , index=False)